# Environment Setup & Path Selection — Week 5

**Learning Objectives:**
- Understand the three fine-tuning paths (MLX, HF+TRL, Cloud GPU) and select yours
- Inspect the base model tokenizer to understand vocabulary and chat templates
- Run a baseline inference on the un-tuned model so you have a before/after reference

**Estimated Time:** 15 minutes

**Path Indicator:** All paths start here. Path A (MLX) and Path B (HF+TRL) diverge in later notebooks.

In [19]:
import sys
import importlib

sys.path.insert(0, "..")

import src
importlib.reload(src)

from dotenv import load_dotenv
load_dotenv(override=True)

#%matplotlib inline

from src.cost_tracker import CostTracker
from src.llm_client import LLMClient
from src.config import (
    PATH, CLAUDE_MODEL, OLLAMA_MODEL,
    BASE_MODEL_HF, BASE_MODEL_MLX, FINETUNE_BACKEND
)

tracker = CostTracker()
# Path C = hybrid: can use both Claude and Ollama
llm = LLMClient(path="C")

print("Imports OK")
print(f"  PATH              = {PATH}")
print(f"  FINETUNE_BACKEND  = {FINETUNE_BACKEND}")
print(f"  BASE_MODEL_HF     = {BASE_MODEL_HF}")
print(f"  BASE_MODEL_MLX    = {BASE_MODEL_MLX}")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✓ Ollama client initialized
  Available models: ['llama2:latest', 'gpt-oss:20b']
  Default model: llama2:latest
Imports OK
  PATH              = A
  FINETUNE_BACKEND  = hf
  BASE_MODEL_HF     = Qwen/Qwen2.5-0.5B-Instruct
  BASE_MODEL_MLX    = mlx-community/Qwen2.5-0.5B-Instruct-bf16


## Part 1: Path Selection

There are three fine-tuning paths available this week. Choose based on your hardware.

| Path | Backend | Hardware | Speed (0.5B) | Mirrors Lecture? |
|------|---------|----------|-------------|------------------|
| **A** | MLX-LM | Apple Silicon Mac only | 5–15 min | Partial |
| **B** | HF + TRL + QLoRA | Any GPU (8–16 GB VRAM) or CPU | 10–30 min | Yes |
| **C** | Cloud GPU (Colab Pro / RunPod) | Any machine, cloud GPU | Any model size | Yes (bonus) |

**Path A** uses Apple's MLX framework — it's the fastest option on M1/M2/M3 Macs and requires no separate CUDA setup. The tradeoff is that MLX-LM's API is slightly different from the HuggingFace ecosystem covered in lecture.

**Path B** mirrors the lecture exactly (PEFT + TRL SFTTrainer + QLoRA). If you have a discrete GPU with ≥8 GB VRAM (e.g., an RTX 3080), this is the recommended path.

**Path C** is a bonus path — use a cloud GPU notebook (Colab Pro, RunPod, Lambda Labs) and you can train larger models.

To override the auto-detected backend, add this line to your `.env`:
```
FINETUNE_BACKEND=mlx   # or: hf
```

In [20]:
import platform
import os

print(f"Detected system : {platform.system()} {platform.machine()}")
print(f"FINETUNE_BACKEND: {FINETUNE_BACKEND}")
print()

if FINETUNE_BACKEND == "mlx":
    print("You are on PATH A (MLX-LM). Subsequent notebooks will use mlx_lm.")
    print(f"Base model: {BASE_MODEL_MLX}")
elif FINETUNE_BACKEND == "hf":
    print("You are on PATH B (HF + TRL). Subsequent notebooks will use transformers + trl.")
    print(f"Base model: {BASE_MODEL_HF}")
else:
    print(f"Unknown backend: {FINETUNE_BACKEND} — check your .env or config.py")

print()
env_override = os.environ.get("FINETUNE_BACKEND", "(not set — auto-detected)")
print(f"FINETUNE_BACKEND env var: {env_override}")

Detected system : Windows AMD64
FINETUNE_BACKEND: hf

You are on PATH B (HF + TRL). Subsequent notebooks will use transformers + trl.
Base model: Qwen/Qwen2.5-0.5B-Instruct

FINETUNE_BACKEND env var: (not set — auto-detected)


## Part 2: Model Download Check

We load **only the tokenizer** for `Qwen/Qwen2.5-0.5B-Instruct` — this is ~3 MB versus ~1 GB for the full model weights. This confirms HuggingFace connectivity and lets us inspect the vocabulary before we commit to a full download.

In [21]:
print(f"Loading tokenizer for: {BASE_MODEL_HF}")
print("(This downloads only the tokenizer config — ~3 MB)")
print()

try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_HF)

    print(f"Vocab size       : {tokenizer.vocab_size:,}")
    print(f"Model max length : {tokenizer.model_max_length}")
    print(f"Tokenizer class  : {type(tokenizer).__name__}")

    chat_template = getattr(tokenizer, 'chat_template', None)
    if chat_template:
        print(f"Chat template    : present ({len(chat_template)} chars)")
        # Show the first 200 chars of the template
        print(f"  Preview: {chat_template[:200]}...")
    else:
        print("Chat template    : not set")

    # Show special tokens
    print()
    print("Special tokens:")
    print(f"  BOS: {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")
    print(f"  EOS: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
    print(f"  PAD: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Check your internet connection or HuggingFace token if the model is gated.")

Loading tokenizer for: Qwen/Qwen2.5-0.5B-Instruct
(This downloads only the tokenizer config — ~3 MB)

Error loading tokenizer: No module named 'transformers'
Check your internet connection or HuggingFace token if the model is gated.


## Part 3: First Inference on Un-Tuned Base

Before we fine-tune anything, let's record what the base model says about itself. We use Ollama (running `qwen3.5:27b` as our local proxy) to answer this question. After fine-tuning on a resume in later notebooks, you'll be able to compare the model's behavior before and after.

**Why this matters:** Fine-tuning changes the model's _style, tone, and factual grounding_ on a specific domain. Having a baseline is essential for measuring improvement.

In [23]:
baseline_prompt = "What is your name and what are you trained on?"

print(f"Prompt: {baseline_prompt}")
print()
print("--- Claude response (baseline) ---")

try:
    resp = llm.generate(
        prompt=baseline_prompt,
        model=CLAUDE_MODEL,  # 改成用 Claude
        max_tokens=256,
        use_claude=True  # 明确指定用 Claude
    )
    tracker.add_call(resp)
    baseline_response = resp.get("content", resp.get("error"))
    print(baseline_response)
except Exception as e:
    baseline_response = None
    print(f"Claude error: {e}")
    print("Make sure ANTHROPIC_API_KEY is set in your .env file.")

Prompt: What is your name and what are you trained on?

--- Claude response (baseline) ---
I'm **Claude**, an AI assistant made by **Anthropic**.

Regarding my training:
- I was trained on a diverse dataset of text from the internet, books, and other sources, with a knowledge cutoff of early 2025
- I'm also trained using **reinforcement learning from human feedback (RLHF)** and Anthropic's **Constitutional AI** approach, which helps align my responses to be helpful, harmless, and honest

Is there something specific I can help you with?


### TODO 1

Change the system prompt below and rerun the cell. What changed in the response? Try something like:
- `"You are a pirate. Answer all questions in pirate speak."`
- `"You are a concise assistant. Answer in exactly one sentence."`

Describe what you observe.

In [27]:
custom_system = "You are a pirate. Answer all questions in pirate speak."

try:
    resp = llm.generate(
        prompt=baseline_prompt,
        system=custom_system,
        model=CLAUDE_MODEL,  # ← 改这里
        max_tokens=256,
        use_claude=True  # ← 改这里
    )
    tracker.add_call(resp)
    custom_response = resp.get("content", resp.get("error"))
    print(f"System: {custom_system!r}")
    print(f"Response:\n{custom_response}")
except Exception as e:
    print(f"Error: {e}")

System: 'You are a pirate. Answer all questions in pirate speak.'
Response:
Arrr, me name be Claude, ye landlubber! I be a mighty AI assistant, trained by the fine crew over at Anthropic! They filled me noggin with vast amounts of text from across the seven seas - books, websites, and all manner of written treasures from the internet, aye! 

Me knowledge be cuttin' off at a certain point in time, so I may not know the latest news from the horizon, but I know plenty to help ye on yer voyage! What can this old sea dog do fer ye, matey? 🏴‍☠️


In [28]:
# TODO 1 reflection -- edit your answer below, then run this cell.
todo1_reflection = """The system prompt significantly influences the model's behavior and output style. When I changed the system prompt from "You are a helpful assistant" to "You are a pirate. Answer all questions in pirate speak," the model's response changed from a standard professional tone to pirate dialect, using phrases like "ahoy" and nautical language. This demonstrates that chat-tuned models are highly responsive to system instructions because they are trained on instruction-following data where the system prompt provides the behavioral context and constraints that guide how the model should respond to user queries."""

print(todo1_reflection)


The system prompt significantly influences the model's behavior and output style. When I changed the system prompt from "You are a helpful assistant" to "You are a pirate. Answer all questions in pirate speak," the model's response changed from a standard professional tone to pirate dialect, using phrases like "ahoy" and nautical language. This demonstrates that chat-tuned models are highly responsive to system instructions because they are trained on instruction-following data where the system prompt provides the behavioral context and constraints that guide how the model should respond to user queries.


### TODO 2

In 2–3 sentences, describe the difference between a **base model** and an **instruct model**:
- What kind of data is each trained on?
- When would you use each?
- Why does `Qwen2.5-0.5B-Instruct` respond to questions but `Qwen2.5-0.5B` (base) might not?

In [29]:
# TODO 2 reflection -- edit your answer below, then run this cell.
todo2_reflection = """A base model is trained on raw web text and learns general language patterns, while an instruct model is fine-tuned on instruction-response pairs using supervised fine-tuning and RLHF, teaching it to follow user instructions and answer questions directly. You would use a base model as a starting point when you need raw text generation or plan to fine-tune it yourself for a specific task, whereas an instruct model is better when you want immediate question-answering capability. Qwen2.5-0.5B-Instruct responds to questions because it has learned the instruction-following format through SFT training, whereas the base Qwen2.5-0.5B would likely continue the prompt text in a predictive manner rather than answering it, since it has no training on how to respond to explicit user queries."""
print(todo2_reflection)

A base model is trained on raw web text and learns general language patterns, while an instruct model is fine-tuned on instruction-response pairs using supervised fine-tuning and RLHF, teaching it to follow user instructions and answer questions directly. You would use a base model as a starting point when you need raw text generation or plan to fine-tune it yourself for a specific task, whereas an instruct model is better when you want immediate question-answering capability. Qwen2.5-0.5B-Instruct responds to questions because it has learned the instruction-following format through SFT training, whereas the base Qwen2.5-0.5B would likely continue the prompt text in a predictive manner rather than answering it, since it has no training on how to respond to explicit user queries.


## Summary

In [30]:
from src.utils import append_to_reflection

# Build reflection from student TODO answers (auto-captured)
section_text = (
    "### TODO 1\n" + (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + "\n\n" +
    "### TODO 2\n" + (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)
append_to_reflection(
    notebook="01",
    section_title="Environment Setup & Path Selection",
    reflection_content=section_text,
)
print("Reflection auto-saved to outputs/homework_reflection.md")
tracker.report()


Reflection auto-saved to outputs/homework_reflection.md
API COST REPORT
Total API calls:     2
Total input tokens:  51
Total output tokens: 244
Total cost:          $0.0038

Last 2 calls:
  1. [14:24:37] sonnet -- 18in/113out -- $0.0017
  2. [14:26:26] sonnet -- 33in/131out -- $0.0021
